<a href="https://colab.research.google.com/github/m4nn4t/stock_price_prediction/blob/main/Multistock_dirAcc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance ta tensorflow scikit-learn matplotlib seaborn -q

  Preparing metadata (setup.py) ... done


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf

In [3]:
stocks=[
    "AAPL",
    "MSFT",
    "GOOGL",
    "AMZN",
    "META",
    "NVDA",
    "TSLA",
    "AMD",
    "NFLX",
    "INTC"
]

In [4]:
all_data=[]

In [5]:
for ticker in stocks:

    df=yf.download(
        ticker,
        start="2018-01-01",
        end="2026-06-05",
        auto_adjust=True,
        progress=False
    )

    if isinstance(df.columns,pd.MultiIndex):
        df.columns=df.columns.get_level_values(0)

    df=df.reset_index()

    df["Ticker"]=ticker

    all_data.append(df)

In [6]:
feature_df=pd.concat(
    all_data,
    ignore_index=True
)

print(feature_df.shape)

(21170, 7)


In [9]:
market_tickers=[
    "SPY",
    "QQQ",
    "^VIX"
]

market={}

In [10]:
for ticker in market_tickers:

    df=yf.download(
        ticker,
        start="2018-01-01",
        end="2026-06-05",
        auto_adjust=True,
        progress=False
    )

    if isinstance(df.columns,pd.MultiIndex):
        df.columns=df.columns.get_level_values(0)

    df=df.reset_index()

    market[ticker]=df

In [11]:
spy=market["SPY"].copy()

spy["SPY_CLOSE"]=spy["Close"]

spy["SPY_RETURN"]=spy["Close"].pct_change()

spy=spy[
    [
        "Date",
        "SPY_CLOSE",
        "SPY_RETURN"
    ]
]

In [12]:
qqq=market["QQQ"].copy()

qqq["QQQ_CLOSE"]=qqq["Close"]

qqq["QQQ_RETURN"]=qqq["Close"].pct_change()

qqq=qqq[
    [
        "Date",
        "QQQ_CLOSE",
        "QQQ_RETURN"
    ]
]

In [13]:
vix=market["^VIX"].copy()

vix["VIX_CLOSE"]=vix["Close"]

vix["VIX_RETURN"]=vix["Close"].pct_change()

vix=vix[
    [
        "Date",
        "VIX_CLOSE",
        "VIX_RETURN"
    ]
]

In [14]:
feature_df["Date"]=pd.to_datetime(
    feature_df["Date"]
).dt.tz_localize(None)

spy["Date"]=pd.to_datetime(
    spy["Date"]
).dt.tz_localize(None)

qqq["Date"]=pd.to_datetime(
    qqq["Date"]
).dt.tz_localize(None)

vix["Date"]=pd.to_datetime(
    vix["Date"]
).dt.tz_localize(None)

In [15]:
feature_df=feature_df.merge(
    spy,
    on="Date",
    how="left"
)

feature_df=feature_df.merge(
    qqq,
    on="Date",
    how="left"
)

feature_df=feature_df.merge(
    vix,
    on="Date",
    how="left"
)

In [16]:
feature_df=feature_df.sort_values(
    ["Ticker","Date"]
)

In [17]:
def create_features(df):

    df=df.copy()

    df["EMA_10"]=df["Close"].ewm(
        span=10,
        adjust=False
    ).mean()

    df["EMA_30"]=df["Close"].ewm(
        span=30,
        adjust=False
    ).mean()

    delta=df["Close"].diff()

    gain=delta.clip(lower=0)

    loss=-delta.clip(upper=0)

    avg_gain=gain.rolling(14).mean()

    avg_loss=loss.rolling(14).mean()

    rs=avg_gain/avg_loss

    df["RSI"]=100-(100/(1+rs))

    ema12=df["Close"].ewm(
        span=12,
        adjust=False
    ).mean()

    ema26=df["Close"].ewm(
        span=26,
        adjust=False
    ).mean()

    df["MACD"]=ema12-ema26

    sma20=df["Close"].rolling(20).mean()

    std20=df["Close"].rolling(20).std()

    df["BB_UPPER"]=sma20+2*std20

    df["BB_LOWER"]=sma20-2*std20

    high_low=df["High"]-df["Low"]

    high_close=np.abs(
        df["High"]-df["Close"].shift()
    )

    low_close=np.abs(
        df["Low"]-df["Close"].shift()
    )

    tr=pd.concat(
        [
            high_low,
            high_close,
            low_close
        ],
        axis=1
    ).max(axis=1)

    df["ATR"]=tr.rolling(14).mean()

    df["RETURN_1D"]=df["Close"].pct_change()

    df["RETURN_5D"]=df["Close"].pct_change(5)

    df["MOMENTUM_10"]=(
        df["Close"]
        -
        df["Close"].shift(10)
    )

    df["VOLATILITY_10"]=(
        df["RETURN_1D"]
        .rolling(10)
        .std()
    )

    return df

In [18]:
feature_df=feature_df.groupby(
    "Ticker",
    group_keys=False
).apply(
    create_features
)

/tmp/ipykernel_1059/925436761.py:70: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["RETURN_1D"]=df["Close"].pct_change()
/tmp/ipykernel_1059/925436761.py:72: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["RETURN_5D"]=df["Close"].pct_change(5)
/tmp/ipykernel_1059/2435774505.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to

In [19]:
feature_df=feature_df.dropna()

print(feature_df.shape)

(20979, 24)


In [20]:
encoder=LabelEncoder()

feature_df["Stock_ID"]=encoder.fit_transform(
    feature_df["Ticker"]
)

In [21]:
print(feature_df.shape)

print(feature_df.columns)

(20979, 25)
Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker', 'SPY_CLOSE',
       'SPY_RETURN', 'QQQ_CLOSE', 'QQQ_RETURN', 'VIX_CLOSE', 'VIX_RETURN',
       'EMA_10', 'EMA_30', 'RSI', 'MACD', 'BB_UPPER', 'BB_LOWER', 'ATR',
       'RETURN_1D', 'RETURN_5D', 'MOMENTUM_10', 'VOLATILITY_10', 'Stock_ID'],
      dtype='object', name='Price')


In [22]:
FEATURE_COLUMNS=[

"Open",
"High",
"Low",
"Close",
"Volume",

"EMA_10",
"EMA_30",

"RSI",

"MACD",

"BB_UPPER",
"BB_LOWER",

"ATR",

"RETURN_1D",
"RETURN_5D",

"MOMENTUM_10",

"VOLATILITY_10",

"SPY_CLOSE",
"SPY_RETURN",

"QQQ_CLOSE",
"QQQ_RETURN",

"VIX_CLOSE",
"VIX_RETURN",

"Stock_ID"

]

In [23]:
from sklearn.preprocessing import MinMaxScaler

scaler=MinMaxScaler()

feature_df[FEATURE_COLUMNS]=scaler.fit_transform(
    feature_df[FEATURE_COLUMNS]
)

In [24]:
WINDOW=100

X_train=[]
y_train=[]

X_test=[]
y_test=[]

In [25]:
for ticker in feature_df["Ticker"].unique():

    stock_df=feature_df[
        feature_df["Ticker"]==ticker
    ].sort_values("Date")

    split_idx=int(
        len(stock_df)*0.8
    )

    train_df=stock_df.iloc[:split_idx]

    test_df=stock_df.iloc[split_idx:]

    train_values=train_df[
        FEATURE_COLUMNS
    ].values

    test_values=test_df[
        FEATURE_COLUMNS
    ].values

    close_idx=FEATURE_COLUMNS.index(
        "Close"
    )

    # TRAIN

    for i in range(
        len(train_values)-WINDOW-1
    ):

        window=train_values[
            i:i+WINDOW
        ]

        current_close=window[
            -1,
            close_idx
        ]

        future_close=train_values[
            i+WINDOW,
            close_idx
        ]

        future_return=(
            future_close
            -
            current_close
        )/current_close

        X_train.append(window)

        y_train.append(
            future_return
        )

    # TEST

    for i in range(
        len(test_values)-WINDOW-1
    ):

        window=test_values[
            i:i+WINDOW
        ]

        current_close=window[
            -1,
            close_idx
        ]

        future_close=test_values[
            i+WINDOW,
            close_idx
        ]

        future_return=(
            future_close
            -
            current_close
        )/current_close

        X_test.append(window)

        y_test.append(
            future_return
        )

/tmp/ipykernel_1059/460514389.py:47: RuntimeWarning: divide by zero encountered in scalar divide
  future_return=(


In [26]:
X_train=np.array(X_train)

y_train=np.array(y_train)

X_test=np.array(X_test)

y_test=np.array(y_test)

print(X_train.shape)

print(y_train.shape)

print(X_test.shape)

print(y_test.shape)

(15769, 100, 23)
(15769,)
(3190, 100, 23)
(3190,)


In [27]:
def create_labels(arr):

    labels=np.where(
        arr>0.01,
        2,
        np.where(
            arr<-0.01,
            0,
            1
        )
    )

    return labels

In [28]:
y_train_mc=create_labels(
    y_train
)

y_test_mc=create_labels(
    y_test
)

In [29]:
unique,counts=np.unique(
    y_train_mc,
    return_counts=True
)

for u,c in zip(unique,counts):

    print(
        f"Class {u}: {c}"
    )

Class 0: 4500
Class 1: 6062
Class 2: 5207


In [30]:
BATCH=64

train_ds=tf.data.Dataset.from_tensor_slices(
    (
        X_train,
        y_train_mc
    )
).shuffle(
    5000
).batch(
    BATCH
).prefetch(
    tf.data.AUTOTUNE
)

test_ds=tf.data.Dataset.from_tensor_slices(
    (
        X_test,
        y_test_mc
    )
).batch(
    BATCH
).prefetch(
    tf.data.AUTOTUNE
)

In [31]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU,Dense,Dropout

In [32]:
classifier=Sequential()

classifier.add(
    GRU(
        128,
        return_sequences=True,
        input_shape=(100,23)
    )
)

classifier.add(
    Dropout(0.1)
)

classifier.add(
    GRU(
        64
    )
)

classifier.add(
    Dense(
        64,
        activation="relu"
    )
)

classifier.add(
    Dense(
        3,
        activation="softmax"
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [33]:
classifier.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

In [34]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [35]:
callbacks=[

    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5
    )

]

In [36]:
history=classifier.fit(

    train_ds,

    validation_data=test_ds,

    epochs=50,

    callbacks=callbacks,

    verbose=1

)

Epoch 1/50
247/247 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.4240 - loss: 1.0695 - val_accuracy: 0.4317 - val_loss: 1.0794 - learning_rate: 0.0010
Epoch 2/50
247/247 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.4316 - loss: 1.0587 - val_accuracy: 0.4398 - val_loss: 1.0646 - learning_rate: 0.0010
Epoch 3/50
247/247 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.4368 - loss: 1.0561 - val_accuracy: 0.4370 - val_loss: 1.0599 - learning_rate: 0.0010
Epoch 4/50
247/247 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.4408 - loss: 1.0558 - val_accuracy: 0.4351 - val_loss: 1.0574 - learning_rate: 0.0010
Epoch 5/50
247/247 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.4449 - loss: 1.0519 - val_accuracy: 0.4423 - val_loss: 1.0584 - learning_rate: 0.0010
Epoch 6/50
247/247 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.4431 - loss: 1.0501 - val_accuracy: 0.4345 - val_loss: 1.0544 - learning_rate: 0.0010
Epoch 7/50
247/247 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.4479 - loss: 1

In [37]:
y_prob=classifier.predict(
    X_test
)

y_pred=np.argmax(
    y_prob,
    axis=1
)

100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


In [38]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_mc,
        y_pred,
        target_names=[
            "DOWN",
            "NEUTRAL",
            "UP"
        ]
    )
)

              precision    recall  f1-score   support

        DOWN       0.36      0.15      0.22       884
     NEUTRAL       0.49      0.73      0.58      1292
          UP       0.40      0.34      0.37      1014

    accuracy                           0.45      3190
   macro avg       0.41      0.41      0.39      3190
weighted avg       0.42      0.45      0.41      3190



In [39]:
print(
    "Accuracy:",
    np.mean(
        y_pred==y_test_mc
    )*100
)

Accuracy: 44.70219435736677
